# Apache Avro — the row-based, schema-with-data format

*Part of the weyland data-format notebook series (B81). Runs entirely offline: no Kafka, no schema
registry, no network. Everything you need — the schema and the records — is defined right here.*

---

## What Avro is

**Apache Avro** is a compact, binary, **row-oriented** serialization format. Its defining trait is that
the **schema travels with the data**: an Avro *container file* embeds its writer schema in a header, so
any reader can decode the bytes without being told the layout out-of-band. The format is
**self-describing**.

That single design choice is what makes Avro the workhorse of streaming and change-data-capture (CDC)
pipelines:

- **Row-based, not columnar.** Avro lays out one whole record after another (all fields of record 1,
  then all fields of record 2, ...). That is exactly the shape a message queue wants — you produce and
  consume *one event at a time*. Compare with **Parquet / Arrow / Lance**, which are *columnar*: they
  group all values of a column together to make analytical scans and compression efficient, at the cost
  of needing the whole row group before you can materialize a record.
- **Schema-with-data → schema evolution.** Because a reader can compare the schema the data was *written*
  with against the schema it *expects*, Avro can resolve differences field-by-field. Add a field with a
  default, drop a field, widen a type — old and new readers keep working. This is the headline feature
  and it gets its own section below.
- **No per-value tags in the payload.** Unlike JSON (every field name repeated on every record) or
  Protobuf (field numbers inline), Avro writes *only the values*, in schema order. The schema is stored
  **once** in the file header. That is why Avro payloads are so small.

### Where Avro sits in the format landscape

| Format | Layout | Schema | Best at |
|---|---|---|---|
| **Avro** | row | with data (self-describing), evolvable | streaming messages, Kafka, CDC, row-at-a-time write/replay |
| **Parquet** | columnar | in footer | analytical scans over cold data on object storage |
| **Arrow** | columnar | in-memory IPC | zero-copy in-memory analytics, engine interchange |
| **Lance** | columnar | on disk | vector search + random access + versioning |
| **JSON** | row | none (or external) | human-readable interchange, debugging |

The mental model: **Avro for transport and evolution, columnar for scan.** A typical lab pipeline lands
Avro/Kafka events, then rewrites them to Parquet/Iceberg for analytics. This notebook walks both halves.


## 0. Setup

We use `fastavro` (a fast, pure-Python Avro implementation — no JVM), plus `pyarrow`, `polars`,
`pandas`, and `numpy` for the columnar bridge at the end. Everything writes to a temp directory that is
cleaned up when the kernel exits.


In [1]:
import io
import json
import os
import tempfile

import fastavro
import numpy as np
import pyarrow as pa
import polars as pl
import pandas as pd

print("fastavro", fastavro.__version__)
print("pyarrow ", pa.__version__)
print("polars  ", pl.__version__)
print("pandas  ", pd.__version__)
print("numpy   ", np.__version__)

# A scratch directory for the .avro files we write. Self-contained; removed on kernel exit.
WORKDIR = tempfile.mkdtemp(prefix="avro_deepdive_")
print("workdir:", WORKDIR)

fastavro 1.12.2
pyarrow  25.0.0
polars   1.44.1
pandas   2.3.3
numpy    2.4.6
workdir: /tmp/avro_deepdive_wc9qmsj0


## 1. Define an Avro schema and sample records — in the notebook

An Avro schema is just JSON. The top-level type here is a `record` (Avro's struct), with named,
typed `fields`. A few things worth noticing:

- **`namespace` + `name`** give the record a fully-qualified name (`weyland.sensor.Reading`). This is
  what a schema registry keys on in a real Kafka deployment.
- **`["null", "double"]`** is a **union**. Avro has no implicit nullability — a field is nullable only
  if `null` is one of its union branches. By convention `null` is listed first and paired with a
  `"default": null`.
- **`logicalType`** annotates a primitive with higher-level meaning. `timestamp-micros` is a `long`
  holding microseconds since the Unix epoch; `fastavro` will convert `datetime` objects for us.
- **`enum`** constrains a field to a fixed symbol set.

This is our **v1** schema — the contract the first producer ships.

In [2]:
schema_v1_dict = {
    "type": "record",
    "namespace": "weyland.sensor",
    "name": "Reading",
    "doc": "A single environmental sensor reading from a lab node (schema v1).",
    "fields": [
        {"name": "reading_id", "type": "long", "doc": "Monotonic id per sensor."},
        {"name": "sensor", "type": "string", "doc": "Sensor hostname, e.g. 'mother' or 'rogueone'."},
        {
            "name": "metric",
            "type": {
                "type": "enum",
                "name": "Metric",
                "symbols": ["TEMPERATURE", "HUMIDITY", "POWER_WATTS"],
            },
        },
        {"name": "value", "type": "double"},
        # Nullable field: a union of null + double, defaulting to null.
        {"name": "calibration_offset", "type": ["null", "double"], "default": None},
        {
            "name": "recorded_at",
            "type": {"type": "long", "logicalType": "timestamp-micros"},
            "doc": "Event time (microseconds since epoch).",
        },
    ],
}

# fastavro wants a *parsed* schema object for writing/reading.
schema_v1 = fastavro.parse_schema(schema_v1_dict)
print("Parsed schema name:", schema_v1["name"])
print(json.dumps(schema_v1_dict, indent=2))

Parsed schema name: weyland.sensor.Reading
{
  "type": "record",
  "namespace": "weyland.sensor",
  "name": "Reading",
  "doc": "A single environmental sensor reading from a lab node (schema v1).",
  "fields": [
    {
      "name": "reading_id",
      "type": "long",
      "doc": "Monotonic id per sensor."
    },
    {
      "name": "sensor",
      "type": "string",
      "doc": "Sensor hostname, e.g. 'mother' or 'rogueone'."
    },
    {
      "name": "metric",
      "type": {
        "type": "enum",
        "name": "Metric",
        "symbols": [
          "TEMPERATURE",
          "HUMIDITY",
          "POWER_WATTS"
        ]
      }
    },
    {
      "name": "value",
      "type": "double"
    },
    {
      "name": "calibration_offset",
      "type": [
        "null",
        "double"
      ],
      "default": null
    },
    {
      "name": "recorded_at",
      "type": {
        "type": "long",
        "logicalType": "timestamp-micros"
      },
      "doc": "Event time (microsecon

Now some sample records. Avro records are plain Python dicts keyed by field name. We build a small,
deterministic batch so the notebook is reproducible.

In [3]:
from datetime import datetime, timezone, timedelta

rng = np.random.default_rng(42)
base_t = datetime(2026, 9, 1, 12, 0, 0, tzinfo=timezone.utc)
sensors = ["mother", "rogueone"]
metrics = ["TEMPERATURE", "HUMIDITY", "POWER_WATTS"]

records_v1 = []
for i in range(1000):
    metric = metrics[i % 3]
    if metric == "TEMPERATURE":
        value = float(round(20 + rng.normal(0, 2), 3))
    elif metric == "HUMIDITY":
        value = float(round(45 + rng.normal(0, 5), 3))
    else:
        value = float(round(180 + rng.normal(0, 30), 3))
    records_v1.append(
        {
            "reading_id": i,
            "sensor": sensors[i % 2],
            "metric": metric,
            "value": value,
            # ~1 in 4 readings carries a calibration offset; the rest are null.
            "calibration_offset": (float(round(rng.normal(0, 0.1), 4)) if i % 4 == 0 else None),
            "recorded_at": base_t + timedelta(seconds=i * 5),
        }
    )

print(f"{len(records_v1)} records built")
records_v1[0]

1000 records built


{'reading_id': 0,
 'sensor': 'mother',
 'metric': 'TEMPERATURE',
 'value': 20.609,
 'calibration_offset': -0.104,
 'recorded_at': datetime.datetime(2026, 9, 1, 12, 0, tzinfo=datetime.timezone.utc)}

### Write an Avro **container file** with `fastavro.writer`

`fastavro.writer(fp, schema, records)` writes an *Object Container File*: a magic header, the
**writer schema embedded as JSON**, then one or more compressed *blocks* of row-packed records, with
sync markers between blocks. Because the schema is in the header, this file is fully self-describing —
hand it to anyone and they can read it.

In [4]:
v1_path = os.path.join(WORKDIR, "readings_v1.avro")

with open(v1_path, "wb") as fp:
    fastavro.writer(fp, schema_v1, records_v1)

size = os.path.getsize(v1_path)
print(f"wrote {len(records_v1)} records -> {v1_path}")
print(f"file size: {size:,} bytes ({size / len(records_v1):.1f} bytes/record)")

wrote 1000 records -> /tmp/avro_deepdive_wc9qmsj0/readings_v1.avro
file size: 30,723 bytes (30.7 bytes/record)


### Read it back — and watch the reader recover the schema

The reader was told *nothing* about the layout. It reads the embedded schema straight out of the file
header. `reader.writer_schema` is that recovered schema — proof the data is self-describing.

In [5]:
with open(v1_path, "rb") as fp:
    reader = fastavro.reader(fp)
    recovered_schema = reader.writer_schema  # <-- pulled from the file header, not supplied by us
    read_back = list(reader)

print("Reader recovered the writer schema from the file header:")
print("  record name :", recovered_schema["name"])
print("  namespace   :", recovered_schema.get("namespace"))
print("  field names :", [f["name"] for f in recovered_schema["fields"]])
print()
print(f"read {len(read_back)} records back")
print("first record:", read_back[0])

# Round-trip sanity: the value we wrote is the value we read (timestamps come back as datetimes).
assert read_back[0]["reading_id"] == 0
assert read_back[500]["metric"] == records_v1[500]["metric"]
print("\nround-trip OK")

Reader recovered the writer schema from the file header:
  record name : weyland.sensor.Reading
  namespace   : None
  field names : ['reading_id', 'sensor', 'metric', 'value', 'calibration_offset', 'recorded_at']

read 1000 records back
first record: {'reading_id': 0, 'sensor': 'mother', 'metric': 'TEMPERATURE', 'value': 20.609, 'calibration_offset': -0.104, 'recorded_at': datetime.datetime(2026, 9, 1, 12, 0, tzinfo=datetime.timezone.utc)}

round-trip OK


## 2. Schema evolution — the headline Avro feature

Here is what Avro is *for*. In a streaming system, producers and consumers are deployed independently
and at different times. The schema of your events **will** change. Avro's answer is
**reader-vs-writer schema resolution**: when the schema the data was written with differs from the
schema the reader expects, Avro reconciles them **field-by-field** using well-defined rules.

Two directions of compatibility matter:

- **Backward compatible** — a **new reader** can read **old data**. (You upgrade the consumer first.)
- **Forward compatible** — an **old reader** can read **new data**. (You upgrade the producer first.)

The rules that make evolution safe:

| Change | Backward? | Forward? | Why |
|---|---|---|---|
| **Add a field *with a default*** | ✅ | ✅ | new reader fills the default when reading old data; old reader ignores the unknown new field |
| Add a field *without* a default | ❌ | ✅ | new reader has nothing to fill for old data |
| **Remove a field** (that had a default) | ✅ | ✅ | new reader ignores it in old data; old reader supplies the default for new data |
| Make a field nullable (`T` → `["null", T]` with default null) | ✅ | ✅ | union widening with a default |
| Rename a field | ❌ | ❌ | resolved by name; use `aliases` instead |

The golden rule: **every field you might ever add or remove should have a `default`.** That one habit
is what keeps a pipeline evolvable.

Below we define a **v2** schema that:
1. **adds** `unit` (a string) *with a default* — safe,
2. **adds** `quality` (nullable enum, default null) — safe,
3. **removes** `calibration_offset` — safe because v1 gave it a default.

`recorded_at` stays (v1 declared it with no default, so dropping it would break an old reader — a good
illustration of *why the golden rule matters*). Then we read the **v1-written file** with the **v2**
reader schema and watch resolution happen.

In [6]:
schema_v2_dict = {
    "type": "record",
    "namespace": "weyland.sensor",
    "name": "Reading",
    "doc": "Environmental sensor reading (schema v2): adds unit + quality, drops calibration_offset.",
    "fields": [
        {"name": "reading_id", "type": "long"},
        {"name": "sensor", "type": "string"},
        {
            "name": "metric",
            "type": {
                "type": "enum",
                "name": "Metric",
                "symbols": ["TEMPERATURE", "HUMIDITY", "POWER_WATTS"],
            },
        },
        {"name": "value", "type": "double"},
        # calibration_offset is GONE in v2 (it had a default in v1, so removal is compatible).
        # NEW field with a default -> backward+forward compatible.
        {"name": "unit", "type": "string", "default": "unknown"},
        # NEW nullable field, default null.
        {
            "name": "quality",
            "type": ["null", {"type": "enum", "name": "Quality", "symbols": ["GOOD", "SUSPECT", "BAD"]}],
            "default": None,
        },
        # recorded_at is KEPT: v1 declared it with no default, so an old reader would have nothing to
        # fill if a new file dropped it. Leaving it in keeps v2 forward-compatible with v1.
        {
            "name": "recorded_at",
            "type": {"type": "long", "logicalType": "timestamp-micros"},
        },
    ],
}

schema_v2 = fastavro.parse_schema(schema_v2_dict)
print("v1 fields:", [f["name"] for f in schema_v1_dict["fields"]])
print("v2 fields:", [f["name"] for f in schema_v2_dict["fields"]])

v1 fields: ['reading_id', 'sensor', 'metric', 'value', 'calibration_offset', 'recorded_at']
v2 fields: ['reading_id', 'sensor', 'metric', 'value', 'unit', 'quality', 'recorded_at']


### Read v1 data with the v2 reader schema

`fastavro.reader(fp, reader_schema=schema_v2)` tells the reader: *the file has whatever writer schema is
in its header, but resolve it against **this** schema.* fastavro does the field-by-field resolution:

- `calibration_offset` is in the file but **not** in v2 → **dropped**.
- `unit` is in v2 but **not** in the file → **filled from its default** `"unknown"`.
- `quality` is in v2 but **not** in the file → **filled from its default** `null`.

In [7]:
with open(v1_path, "rb") as fp:
    reader_v2 = fastavro.reader(fp, reader_schema=schema_v2)
    resolved = list(reader_v2)

sample = resolved[0]
print("v1 record read through the v2 reader schema:")
print(json.dumps({k: (v.isoformat() if hasattr(v, "isoformat") else v) for k, v in sample.items()},
                 indent=2, default=str))

print()
# The default filled in for the added fields:
assert sample["unit"] == "unknown", "added field should be filled from its default"
assert sample["quality"] is None, "added nullable field should default to null"
# The removed field is gone:
assert "calibration_offset" not in sample, "removed field should not appear under the v2 schema"
print("resolution OK: 'unit' -> default 'unknown', 'quality' -> default null, "
      "'calibration_offset' dropped")

v1 record read through the v2 reader schema:
{
  "reading_id": 0,
  "sensor": "mother",
  "metric": "TEMPERATURE",
  "value": 20.609,
  "recorded_at": "2026-09-01T12:00:00+00:00",
  "unit": "unknown",
  "quality": null
}

resolution OK: 'unit' -> default 'unknown', 'quality' -> default null, 'calibration_offset' dropped


### The other direction: old reader, new data (forward compatibility)

Now write a **v2** file and read it back with the **v1** reader schema — simulating an old consumer that
has not been upgraded yet. v1 doesn't know about `unit` or `quality`, so it **ignores** them; and it
expects `calibration_offset`, which v2 dropped, so it **supplies v1's default** (`null`). Old code keeps
running against new data.

In [8]:
# Build a few v2 records (note: they have unit + quality, and no calibration_offset).
records_v2 = [
    {"reading_id": 9000 + i, "sensor": "mother", "metric": "TEMPERATURE",
     "value": 21.5 + i, "unit": "celsius",
     "quality": ["GOOD", "SUSPECT", "BAD"][i % 3],
     "recorded_at": base_t + timedelta(minutes=i)}
    for i in range(5)
]

v2_path = os.path.join(WORKDIR, "readings_v2.avro")
with open(v2_path, "wb") as fp:
    fastavro.writer(fp, schema_v2, records_v2)

# Old (v1) reader consuming new (v2) data:
with open(v2_path, "rb") as fp:
    old_reader = fastavro.reader(fp, reader_schema=schema_v1)
    seen_by_old = list(old_reader)

s = seen_by_old[0]
print("v2 record read through the OLD v1 reader schema:")
print("  kept fields   :", [k for k in s.keys()])
print("  unit/quality  : ignored (v1 doesn't know them)")
print("  calibration_offset ->", s["calibration_offset"], "(v1 default, filled because v2 dropped it)")
assert "unit" not in s and "quality" not in s
assert s["calibration_offset"] is None
print("\nforward-compatibility OK: old reader survives new data")

v2 record read through the OLD v1 reader schema:
  kept fields   : ['reading_id', 'sensor', 'metric', 'value', 'recorded_at', 'calibration_offset']
  unit/quality  : ignored (v1 doesn't know them)
  calibration_offset -> None (v1 default, filled because v2 dropped it)

forward-compatibility OK: old reader survives new data


### An *incompatible* change fails loudly — as it should

Compatibility isn't magic; it follows the rules in the table above. Adding a field **without** a default
and then trying to read **old data** with that schema has no value to supply, and Avro raises rather than
inventing one. Fail-closed is the correct behavior — a silent wrong default would be worse.

In [9]:
schema_bad_dict = {
    "type": "record", "namespace": "weyland.sensor", "name": "Reading",
    "fields": [
        {"name": "reading_id", "type": "long"},
        {"name": "sensor", "type": "string"},
        {"name": "metric", "type": {"type": "enum", "name": "Metric",
                                     "symbols": ["TEMPERATURE", "HUMIDITY", "POWER_WATTS"]}},
        {"name": "value", "type": "double"},
        # NEW required field, NO default -> incompatible with old data.
        {"name": "site", "type": "string"},
    ],
}
schema_bad = fastavro.parse_schema(schema_bad_dict)

failed = False
try:
    with open(v1_path, "rb") as fp:
        list(fastavro.reader(fp, reader_schema=schema_bad))
except Exception as exc:  # fastavro raises SchemaResolutionError
    failed = True
    print(f"reading v1 data with an incompatible schema raised: {type(exc).__name__}")
    print(f"  -> {exc}")

assert failed, "an added field with no default MUST fail against old data"
print("\ngood: the incompatible change failed closed instead of inventing a value")

reading v1 data with an incompatible schema raised: SchemaResolutionError
  -> No default value for field site in weyland.sensor.Reading

good: the incompatible change failed closed instead of inventing a value


## 3. Compression codecs and the compact-binary payoff

An Avro container file compresses each *block* of records with a **codec**. The universally-available
codecs are:

- **`null`** — no compression (baseline).
- **`deflate`** — zlib/DEFLATE, always available (stdlib). Good ratio, moderate CPU.
- **`snappy`** — very fast, lower ratio; needs `python-snappy`/`cramjam`. **Optional** — common in Kafka
  land but not always installed. We **probe for it at runtime** and skip if absent.

We write the same 1000 records under each available codec and compare sizes, then compare against a
plain **JSON** representation of the same records to show how much the schema-once, values-only design
saves.

In [10]:
def try_write(codec):
    """Write records_v1 under `codec`; return byte size, or None if the codec is unavailable."""
    path = os.path.join(WORKDIR, f"readings_{codec}.avro")
    try:
        with open(path, "wb") as fp:
            fastavro.writer(fp, schema_v1, records_v1, codec=codec)
        return os.path.getsize(path)
    except (ValueError, ImportError, Exception) as exc:  # unknown/unavailable codec
        print(f"  codec {codec!r} unavailable: {type(exc).__name__}: {exc}")
        return None

candidate_codecs = ["null", "deflate", "snappy"]
sizes = {}
for c in candidate_codecs:
    s = try_write(c)
    if s is not None:
        sizes[c] = s

print()
for c, s in sizes.items():
    print(f"  {c:>8}: {s:>8,} bytes")

if "snappy" not in sizes:
    print("\nNOTE: 'snappy' is not installed in this image — that's expected and harmless. "
          "Install python-snappy or cramjam to enable it. deflate + null always work.")

  codec 'snappy' unavailable: ValueError: snappy codec is supported but you need to install one of the following libraries: ('cramjam',)

      null:   30,723 bytes
   deflate:   15,602 bytes

NOTE: 'snappy' is not installed in this image — that's expected and harmless. Install python-snappy or cramjam to enable it. deflate + null always work.


### Avro binary vs JSON for the identical records

To make the comparison fair we serialize the records to JSON (converting the `datetime`/enum to plain
values). JSON repeats every field **name** on every one of the 1000 records and stores numbers as text;
Avro stores the field names **once** in the header and the values as packed binary. The ratio below is
why row-based streaming systems reach for Avro over JSON on the wire.

In [11]:
def json_default(o):
    if hasattr(o, "isoformat"):
        return o.isoformat()
    return str(o)

json_blob = json.dumps(records_v1, default=json_default).encode("utf-8")
json_size = len(json_blob)

# A gzipped-JSON point of comparison, to isolate "columnar-free schema economy" from raw compression.
import gzip
json_gzip_size = len(gzip.compress(json_blob))

avro_null = sizes["null"]
avro_deflate = sizes.get("deflate")

print(f"  JSON (raw)        : {json_size:>9,} bytes")
print(f"  JSON (gzip)       : {json_gzip_size:>9,} bytes")
print(f"  Avro (null codec) : {avro_null:>9,} bytes   -> {json_size / avro_null:5.1f}x smaller than raw JSON")
if avro_deflate:
    print(f"  Avro (deflate)    : {avro_deflate:>9,} bytes   -> {json_size / avro_deflate:5.1f}x smaller than raw JSON")

print("\nAvro's win is structural: the schema is stored ONCE in the header, not repeated on every")
print("record, and values are binary-packed instead of rendered as text. That is why the")
print("uncompressed Avro file is already several times smaller than raw JSON.")
if avro_deflate and json_gzip_size < avro_deflate:
    print()
    print(f"Note the honest wrinkle: gzipped JSON ({json_gzip_size:,} B) actually beats Avro+deflate")
    print(f"({avro_deflate:,} B) on THIS small, highly-repetitive batch. Whole-blob gzip dedupes the")
    print("repeated field names brilliantly -- but that redundancy only exists because JSON put the")
    print("names there in the first place. On the wire, per-message (you compress one small event at a")
    print("time, not a 1000-record blob), Avro's schema-once framing is the durable advantage; the")
    print("registry also means the schema isn't even in each message. Compression ratio on a batch is")
    print("not the metric a streaming system optimizes.")

  JSON (raw)        :   155,731 bytes
  JSON (gzip)       :    11,230 bytes
  Avro (null codec) :    30,723 bytes   ->   5.1x smaller than raw JSON
  Avro (deflate)    :    15,602 bytes   ->  10.0x smaller than raw JSON

Avro's win is structural: the schema is stored ONCE in the header, not repeated on every
record, and values are binary-packed instead of rendered as text. That is why the
uncompressed Avro file is already several times smaller than raw JSON.

Note the honest wrinkle: gzipped JSON (11,230 B) actually beats Avro+deflate
(15,602 B) on THIS small, highly-repetitive batch. Whole-blob gzip dedupes the
repeated field names brilliantly -- but that redundancy only exists because JSON put the
names there in the first place. On the wire, per-message (you compress one small event at a
time, not a 1000-record blob), Avro's schema-once framing is the durable advantage; the
registry also means the schema isn't even in each message. Compression ratio on a batch is
not the metric a str

## 4. Bridge to the columnar world — Avro for transport, columnar for scan

Avro is a *transport and evolution* format; it is **not** what you want to run analytical scans over.
For that you materialize the rows into a **columnar** table — Arrow in memory, Parquet/Iceberg on disk —
where per-column layout makes aggregations and predicate pushdown fast.

The pattern in one sentence: **decode Avro rows → build a columnar table → analyze there.** Below we take
the records we read back and load them into both a **polars** DataFrame and a **PyArrow** Table, then run
a group-by that would be efficient on columnar storage.

In [12]:
# Re-read the v1 file into a list of row dicts (this is the "consume the stream" step).
with open(v1_path, "rb") as fp:
    rows = list(fastavro.reader(fp))

# --- polars (row dicts -> columnar DataFrame) ---
df = pl.DataFrame(rows)
print("polars DataFrame:")
print(df.schema)
print(df.head(3))

agg = (
    df.group_by("metric")
      .agg(
          pl.len().alias("n"),
          pl.col("value").mean().round(2).alias("mean_value"),
          pl.col("value").min().alias("min_value"),
          pl.col("value").max().alias("max_value"),
      )
      .sort("metric")
)
print("\nper-metric aggregate (columnar scan):")
print(agg)

polars DataFrame:
Schema({'reading_id': Int64, 'sensor': String, 'metric': String, 'value': Float64, 'calibration_offset': Float64, 'recorded_at': Datetime(time_unit='us', time_zone='UTC')})
shape: (3, 6)
┌────────────┬──────────┬─────────────┬─────────┬────────────────────┬─────────────────────────┐
│ reading_id ┆ sensor   ┆ metric      ┆ value   ┆ calibration_offset ┆ recorded_at             │
│ ---        ┆ ---      ┆ ---         ┆ ---     ┆ ---                ┆ ---                     │
│ i64        ┆ str      ┆ str         ┆ f64     ┆ f64                ┆ datetime[μs, UTC]       │
╞════════════╪══════════╪═════════════╪═════════╪════════════════════╪═════════════════════════╡
│ 0          ┆ mother   ┆ TEMPERATURE ┆ 20.609  ┆ -0.104             ┆ 2026-09-01 12:00:00 UTC │
│ 1          ┆ rogueone ┆ HUMIDITY    ┆ 48.752  ┆ null               ┆ 2026-09-01 12:00:05 UTC │
│ 2          ┆ mother   ┆ POWER_WATTS ┆ 208.217 ┆ null               ┆ 2026-09-01 12:00:10 UTC │
└────────────┴─────

In [13]:
# --- PyArrow (row dicts -> Arrow Table), the interchange format for the columnar ecosystem ---
table = pa.Table.from_pylist(rows)
print("Arrow schema (note columnar types + the timestamp logical type carried through):")
print(table.schema)
print(f"\nArrow table: {table.num_rows:,} rows x {table.num_columns} columns")

# From here you'd typically write Parquet/Iceberg for cold analytics:
#   import pyarrow.parquet as pq; pq.write_table(table, "readings.parquet")
# Same records, now in a layout built for scanning instead of streaming.
sensor_counts = table.group_by("sensor").aggregate([("reading_id", "count")])
print("\nrows per sensor (Arrow group-by):")
print(sensor_counts.to_pydict())

Arrow schema (note columnar types + the timestamp logical type carried through):
reading_id: int64
sensor: string
metric: string
value: double
calibration_offset: double
recorded_at: timestamp[us, tz=UTC]

Arrow table: 1,000 rows x 6 columns

rows per sensor (Arrow group-by):
{'sensor': ['mother', 'rogueone'], 'reading_id_count': [500, 500]}


## 5. When to reach for Avro (and when not to)

**Use Avro when:**

- You are moving **messages / events one at a time** — Kafka topics, Redpanda, a CDC stream, a job queue.
  Row layout matches the access pattern; you rarely scan a whole topic column-wise.
- Your schema **will evolve** and producers/consumers deploy independently. Avro's reader-vs-writer
  resolution with defaults is purpose-built for this, and a schema registry (Confluent, Apicurio, the
  lab's Redpanda registry) enforces compatibility at publish time.
- You want a **compact, self-describing** payload without repeating field names per record (vs JSON) and
  without a separate `.proto` compile step for dynamic consumers (vs Protobuf).
- You need **language-neutral** row records with a canonical binary encoding across the JVM and Python
  ecosystems.

**Reach for a columnar format instead when:**

- **Parquet** — cold, at-rest analytical data on object storage (S3/MinIO, Iceberg tables). Column
  pruning + predicate pushdown + excellent compression. This is where your Avro/Kafka events *land* for
  analytics.
- **Arrow** — in-memory analytics and zero-copy interchange between engines (polars, DuckDB, Trino,
  pandas). The lingua franca *between* systems, not a storage format.
- **Lance** — random access, vector search, and dataset versioning (ML feature/embedding stores).

**Rule of thumb:** *Avro on the wire and in the log; columnar on disk for scan.* A healthy pipeline uses
both — Avro to carry evolving events reliably, then a rewrite to Parquet/Iceberg for the analytical
tier. The other notebooks in this series cover the columnar side.

---

### Recap of what this notebook demonstrated

1. Defined an Avro schema (record + enum + union/nullable + logical timestamp) and sample records **in
   the notebook**.
2. Wrote a self-describing **container file** with `fastavro.writer` and read it back with
   `fastavro.reader`, recovering the schema **from the file header**.
3. **Schema evolution**: read v1 data with a v2 reader schema (added fields filled from defaults, removed
   field dropped), showed the reverse direction (old reader, new data), and showed an incompatible change
   **failing closed**.
4. Compared **codecs** (`null`, `deflate`, and `snappy` when available) and Avro's compact binary against
   raw and gzipped **JSON** of the same records.
5. Bridged to the **columnar** world by loading the decoded rows into **polars** and **PyArrow** for
   analytics — *Avro for transport, columnar for scan.*


### Cleanup

Remove the scratch directory. (Skip this cell if you want to inspect the `.avro` files with an external
tool.)

In [14]:
import shutil
shutil.rmtree(WORKDIR, ignore_errors=True)
print("removed", WORKDIR)

removed /tmp/avro_deepdive_wc9qmsj0
